# Module 2: RAG (Retrieval-Augmented Generation) System

**Duration:** ~45 minutes

## Learning Objectives
- Understand how RAG enhances LLM capabilities
- Process documents and create embeddings
- Build a ChromaDB vector store
- Implement a complete RAG pipeline with LangChain
- Compare RAG-enhanced responses with baseline

## Overview
RAG allows us to augment an LLM with external knowledge without retraining. We'll build a system that retrieves relevant F5 documentation and includes it as context when generating responses.

## 2.1 Install RAG Dependencies

Run the cell below to install dependencies. 

**Important:** After running the install cell, if you get import errors (like `RuntimeError: function 'istft' already has a docstring`), you need to:
1. **Restart runtime**: Runtime > Restart runtime (or press `Ctrl+M .`)
2. **Run all cells again**: Runtime > Run all

This is a known Colab behavior when installing packages that modify PyTorch.

In [ ]:
%%capture
# Install LangChain ecosystem
!pip install -q langchain>=0.3.0
!pip install -q langchain-core>=0.3.0
!pip install -q langchain-community>=0.3.0
!pip install -q langchain-huggingface>=0.1.0
!pip install -q langchain-text-splitters>=0.3.0

# Install vector store and embeddings
!pip install -q chromadb>=1.0.0
!pip install -q sentence-transformers>=3.0.0

# Fix potential sqlite issue in Colab
!pip install -q pysqlite3-binary

# Install model dependencies (needed if not continuing from Module 1)
!pip install -q transformers accelerate
!pip install -q bitsandbytes>=0.49.0

print("Dependencies installed!")

In [ ]:
# ⚠️ IMPORTANT: If you get a "function 'istft' already has a docstring" error,
# you need to restart the runtime:
#   1. Click Runtime > Restart runtime (or Ctrl+M .)
#   2. Then run all cells from the beginning (Runtime > Run all)
#
# This is a known Colab issue when installing packages that modify torch.

# Handle sqlite3 version issue in Colab
import sys
try:
    __import__('pysqlite3')
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

# Verify installations
import langchain
import chromadb
import sentence_transformers

print(f"langchain: {langchain.__version__}")
print(f"chromadb: {chromadb.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print("\n✅ RAG dependencies installed!")

## 2.2 Load the Base Model

If you're continuing from Module 1, the model should still be loaded. Otherwise, we'll reload it.

In [ ]:
import torch
import os

# Clone repo if running in Colab and data doesn't exist
if not os.path.exists('data/f5_docs'):
    if os.path.exists('llm-finetuning-rag-lab'):
        os.chdir('llm-finetuning-rag-lab')
    else:
        print("Cloning repository...")
        !git clone https://github.com/therealnoof/llm-finetuning-rag-lab.git
        os.chdir('llm-finetuning-rag-lab')
        print("✅ Repository cloned!")

print(f"Working directory: {os.getcwd()}")
print(f"GPU available: {torch.cuda.is_available()}")

# Verify data exists
if os.path.exists('data/f5_docs'):
    print(f"✅ F5 docs directory found")
else:
    print("❌ Data directory not found - check repository clone")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Check if model is already loaded
try:
    _ = model.device
    print("✅ Model already loaded")
except NameError:
    print("Loading model...")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("✅ Model loaded")

## 2.3 Understanding RAG Architecture

RAG consists of three main stages:

1. **Indexing**: Convert documents to embeddings and store in vector database
2. **Retrieval**: Find relevant documents for a given query
3. **Generation**: Use retrieved context to generate informed responses

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Documents  │────▶│  Embeddings │────▶│  Vector DB  │
└─────────────┘     └─────────────┘     └──────┬──────┘
                                               │
┌─────────────┐     ┌─────────────┐           │
│   Query     │────▶│  Retrieval  │◀──────────┘
└─────────────┘     └──────┬──────┘
                           │
                    ┌──────▼──────┐     ┌─────────────┐
                    │   Context   │────▶│     LLM     │────▶ Response
                    └─────────────┘     └─────────────┘
```

## 2.4 Load F5 Documentation

Let's load our F5 documentation files that will serve as the knowledge base.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all text files from the F5 docs directory
loader = DirectoryLoader(
    "data/f5_docs/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} documents")
for doc in documents:
    source = doc.metadata.get('source', 'unknown')
    chars = len(doc.page_content)
    print(f"   - {source}: {chars:,} characters")

In [ ]:
# Preview a document
print("📄 Sample document content (first 500 chars):")
print("-" * 50)
print(documents[0].page_content[:500])

## 2.5 Split Documents into Chunks

Large documents need to be split into smaller chunks for effective retrieval. We'll use a recursive character splitter that respects natural text boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Target chunk size in characters
    chunk_overlap=50,    # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Split priority
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

In [ ]:
# Preview some chunks
print("📄 Sample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200] + "...")

## 2.6 Create Embeddings

Embeddings convert text into dense vectors that capture semantic meaning. We'll use a sentence-transformers model that's fast and effective.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded")

In [ ]:
# Test the embeddings
test_text = "F5 BIG-IP virtual server configuration"
test_embedding = embeddings.embed_query(test_text)

print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

## 2.7 Create Vector Store

ChromaDB is a lightweight vector database that runs locally. We'll store our document embeddings here for fast similarity search.

In [ ]:
from langchain_community.vectorstores import Chroma

# Create vector store from chunks
print("Creating vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="f5_docs",
    persist_directory="./chroma_db"
)

print(f"✅ Vector store created with {len(chunks)} chunks")

## 2.8 Test Retrieval

Let's test the retrieval to see what documents are returned for a query.

In [ ]:
# Create a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 matches
)

# Test query
test_query = "How do I configure SSL offloading?"
print(f"Query: {test_query}\n")

retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get('source', 'unknown').split('/')[-1]
    print(f"--- Document {i} (from {source}) ---")
    print(doc.page_content[:300] + "...\n")

## 2.9 Build the RAG Chain

Now we'll combine retrieval with our LLM to create a complete RAG pipeline.

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# Create a text generation pipeline
text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    return_full_text=False
)

# Wrap in LangChain
llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

print("✅ LLM pipeline created")

In [ ]:
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Create a prompt template that includes context
template = """<|system|>
You are an F5 Networks technical expert. Use the following context to answer the question accurately. If the answer is not in the context, say so but still provide helpful information.</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

print("✅ Prompt template created")

In [ ]:
# Create the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Stuffs all retrieved docs into context
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✅ RAG chain created")

In [ ]:
def query_rag(question):
    """
    Query the RAG system and return the response with sources.
    """
    result = rag_chain.invoke({"query": question})
    
    response = result["result"]
    sources = [doc.metadata.get('source', 'unknown').split('/')[-1] 
               for doc in result.get("source_documents", [])]
    
    return response, sources

print("✅ RAG query function ready")

## 2.10 Test RAG System

Let's test the RAG system with our evaluation questions.

In [ ]:
# Test with a single question first
test_question = "How do I configure SSL offloading on F5 BIG-IP?"

print(f"Question: {test_question}\n")
print("Querying RAG system...\n")

response, sources = query_rag(test_question)

print(f"Response:\n{response}\n")
print(f"Sources: {', '.join(set(sources))}")

## 2.11 Evaluate RAG on All Questions

Now let's run all evaluation questions through the RAG system.

In [ ]:
# Same evaluation questions from Module 1
eval_questions = [
    "What is a virtual server in F5 BIG-IP and what is its purpose?",
    "How do I configure SSL offloading on F5 BIG-IP?",
    "Write an iRule to redirect HTTP to HTTPS.",
    "What is the difference between Least Connections and Round Robin load balancing?",
    "How do I troubleshoot a pool member that is marked as offline?"
]

print(f"Evaluating {len(eval_questions)} questions with RAG...\n")
print("=" * 80)

In [ ]:
# Store RAG responses
rag_responses = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print("-" * 80)
    
    response, sources = query_rag(question)
    rag_responses.append({
        "question": question,
        "response": response,
        "sources": list(set(sources))
    })
    
    print(f"Response:\n{response}")
    print(f"\nSources: {', '.join(set(sources))}")

print(f"\n{'='*80}")
print("RAG evaluation complete!")

## 2.12 Compare with Baseline

Let's load the baseline results and compare side-by-side.

In [ ]:
import json

# Load baseline results
try:
    with open("results/baseline_results.json", "r") as f:
        baseline_data = json.load(f)
    baseline_responses = baseline_data["responses"]
    print("✅ Baseline results loaded")
except FileNotFoundError:
    print("⚠️ Baseline results not found. Run Module 1 first.")
    baseline_responses = []

In [ ]:
# Side-by-side comparison
if baseline_responses:
    print("SIDE-BY-SIDE COMPARISON")
    print("=" * 80)
    
    for i, (baseline, rag) in enumerate(zip(baseline_responses, rag_responses), 1):
        print(f"\n{'='*80}")
        print(f"Question {i}: {baseline['question']}")
        
        print(f"\n{'─'*40}")
        print("📦 BASELINE RESPONSE:")
        print(baseline['response'][:400] + "..." if len(baseline['response']) > 400 else baseline['response'])
        
        print(f"\n{'─'*40}")
        print("🔍 RAG RESPONSE:")
        print(rag['response'][:400] + "..." if len(rag['response']) > 400 else rag['response'])
        print(f"   Sources: {', '.join(rag['sources'])}")

## 2.13 Save RAG Results

In [ ]:
import os

os.makedirs("results", exist_ok=True)

# Save RAG results
results = {
    "model": MODEL_NAME,
    "type": "rag",
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": 500,
    "responses": rag_responses
}

with open("results/rag_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ RAG results saved to results/rag_results.json")

## 2.14 Understanding RAG Trade-offs

### Advantages of RAG
✅ **No training required** - Just add documents  
✅ **Easy to update** - Add/remove documents anytime  
✅ **Source attribution** - Know where answers come from  
✅ **Current information** - Not limited by training cutoff  

### Limitations of RAG
⚠️ **Retrieval quality matters** - Bad retrieval = bad answers  
⚠️ **Context window limits** - Can only use limited context  
⚠️ **Latency overhead** - Retrieval adds time  
⚠️ **Doesn't learn patterns** - Can't generalize beyond documents  

### When to Use RAG
- ✅ Frequently updated knowledge bases
- ✅ Need for source attribution
- ✅ Factual/reference queries
- ✅ Limited training data or compute

## 2.15 Summary

In this module, we:

1. ✅ Built a document processing pipeline
2. ✅ Created embeddings using sentence-transformers
3. ✅ Stored vectors in ChromaDB
4. ✅ Implemented a complete RAG chain with LangChain
5. ✅ Compared RAG responses with baseline

### Key Observations
- RAG responses include specific information from documentation
- Source attribution helps verify answer accuracy
- Quality depends heavily on document coverage and retrieval

### Next Steps
In **Module 3**, we'll fine-tune the model itself on F5-specific data using QLoRA, creating a model that has learned F5 domain knowledge.

In [ ]:
# Optional: Clear some memory before Module 3
# Uncomment if needed

# import gc
# del vectorstore
# del rag_chain
# gc.collect()
# torch.cuda.empty_cache()
# print("Memory cleared")